# Fine-tune Llama-3 8B with QLoRA for spend categorization

Run this notebook on **Google Colab with a free T4 GPU**: *Runtime → Change runtime type → T4 GPU*.
Then *Runtime → Run all*. The whole thing takes about **40–60 minutes**.

What it does:
1. Loads the project (upload the zip, or clone it from GitHub)
2. Trains the TF-IDF baseline (for comparison)
3. *(optional)* Measures Llama-3 **zero-shot** accuracy (before fine-tuning)
4. **Fine-tunes Llama-3 8B Instruct with QLoRA + PEFT**
5. Evaluates the fine-tuned model on the held-out test set
6. Categorizes the full 5,000-row dataset with the fine-tuned model (used by the agent/app)
7. Packs the adapter + reports + categorized data into `results.zip` to download

In [ ]:
# 1. Check the GPU (should say Tesla T4 or better)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the project
By default the next cell clones the project from GitHub. To use a local copy instead, set `GITHUB_URL = ""` and upload `spend-categorization-agent.zip` when asked.

In [ ]:
import os, zipfile
GITHUB_URL = "https://github.com/mansi-ambwani/spend-categorization-agent.git"   # set to "" to upload a zip instead

%cd /content
if GITHUB_URL:
    !git clone -q {GITHUB_URL} spend-categorization-agent
elif not os.path.exists("/content/spend-categorization-agent"):
    from google.colab import files
    up = files.upload()                      # choose spend-categorization-agent.zip
    zname = next(iter(up))
    with zipfile.ZipFile(zname) as z:
        z.extractall("/content")
%cd /content/spend-categorization-agent
!ls

In [ ]:
# 3. Install libraries (torch is already on Colab)
!pip install -q -r requirements-train.txt

In [ ]:
# 4. Data: already included in the project. Re-create it if missing (deterministic, same result).
import os
if not os.path.exists("data/processed/train.jsonl"):
    !python -m src.data_generation --n 5000
    !python -m src.prepare_data
!wc -l data/processed/*.jsonl

In [ ]:
# 5. Classical baseline (TF-IDF + Logistic Regression) - a few seconds
!python -m src.baseline

## 6. (Optional) Zero-shot Llama-3 before fine-tuning
Shows how much fine-tuning helps. Uses 300 test rows to save time (~5 min incl. download of the 16 GB model).

In [ ]:
!python -m src.evaluate --model llm --no-adapter --limit 300

## 7. QLoRA fine-tuning
* base model loaded in **4-bit NF4** (bitsandbytes), frozen
* **LoRA** adapters (r=16, α=32) on all attention + MLP projections (PEFT) → only ~0.5% of weights train
* loss only on the answer tokens, 1 epoch, lr 2e-4, cosine schedule, paged 8-bit AdamW

~15–30 min on a T4. If Colab is slow, add `--max-train-samples 1500`.

In [ ]:
!python -m src.train_qlora

In [ ]:
# 8. Evaluate the fine-tuned model on the full test set
!python -m src.evaluate --model llm

In [ ]:
from IPython.display import Image, display
for p in ["reports/training_loss.png", "reports/confusion_llm_qlora.png"]:
    if os.path.exists(p): display(Image(p, width=700))

In [ ]:
# 9. Categorize the whole dataset with the fine-tuned model (feeds the agent / dashboard)  ~10-15 min
!python -m src.categorize --model llm --batch-size 32

In [ ]:
# 10. Quick look at the agent (offline rule-based mode, no key needed)
!python -m src.agent.agent "top 5 suppliers by spend"
!python -m src.agent.agent "monthly trend of software spend"

In [ ]:
# 11. Package results and download them. Unzip results.zip INTO the project folder on your laptop.
!rm -f results.zip
!zip -qr results.zip models/llama3-spend-qlora reports data/outputs
from google.colab import files
files.download("results.zip")